In [ ]:
import sys, os, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Markdown, display
import config
from src.utils import set_seed
set_seed(config.SEED)
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.facecolor": "#F5F1E8", "figure.facecolor": "#F5F1E8", "axes.edgecolor": "#77766D",
                     "font.size": 10, "axes.titleweight": "bold"})
PALETTE = ["#344C3D", "#BD755A", "#C4A46A", "#899982", "#77766D", "#242723"]
sns.set_palette(PALETTE)
def note(txt):  # data-driven observation rendered as markdown
    display(Markdown("> **Observation.** " + txt))

# 03 — Preprocessing and Feature Engineering

**Question.** Which text-normalisation and feature choices actually help a sentence-role classifier on biomedical text — and which "standard" steps hurt?

**Plan**

1. Look at what tokenisers, stopword lists, stemmers and lemmatisers do to *clinical* text (qualitatively).
2. Compare six preprocessing strategies **empirically** (several random training subsets → mean ± std dev-set macro-F1).
3. Compare feature configurations: n-gram range, sublinear TF-IDF, `min_df`, counts vs TF-IDF.
4. Compare **TF-IDF vs Word2Vec document embeddings**.
5. Analyse vocabulary / out-of-vocabulary (OOV) behaviour per strategy.
6. Record the decision for the classical pipeline in `artifacts/classification/preprocessing_choice.json`.

**Leakage rule:** all vectorisers/embeddings are fitted on *training* data only; strategies are selected on the **dev** split; the test split is untouched until notebook 04.
All preprocessing lives in `src/preprocessing.py` and is the *same code* used by the Flask app.

In [ ]:
import re, time
import src.data as D
from collections import Counter
from src.preprocessing import preprocess, tokenize, STOPWORDS, NEGATIONS, STRATEGIES, extract_dosages, split_sentences
from src.features import build_tfidf
from src.evaluation import classification_metrics
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
train, dev = D.load_pubmed_rct("train"), D.load_pubmed_rct("dev")
LAB = config.CLASS_LABELS
print(len(train), len(dev))

## 1. Qualitative look at clinical text

Clinical sentences contain things generic NLP pipelines mishandle: **dosages** (`5 mg/kg`), **comparators** (`p<0.05`), **negations** (`no significant difference`),
**hyphenated compounds** (`double-blind`), **percentages** and **abbreviations** (`RCT`, `BP`).

In [ ]:
examples = [
    "Patients received 5 mg/kg of drug X twice daily vs. placebo (p<0.05).",
    "There was no significant difference in HbA1c between groups (mean 7.2% vs 7.4%).",
    "A double-blind RCT enrolled 120 patients; BP fell by 12.5 mmHg (95% CI 8.1-16.9).",
]
import sklearn.feature_extraction.text as sk
default_tok = sk.TfidfVectorizer().build_analyzer()
rows = []
for s in examples:
    rows.append({"sentence": s, "sklearn default analyzer": default_tok(s), "MediScope tokenize": tokenize(s)})
pd.DataFrame(rows).set_index("sentence").style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

**Interpretation.** The default scikit-learn analyzer silently drops one-character tokens (`p`, `n`, `x`), splits `5 mg/kg` apart from its number context, discards the `<` of `p<0.05`
and breaks decimals (`12.5` → `12`, `5`). MediScope's regex tokeniser keeps decimals and percentages as single tokens, keeps comparison operators and hyphenated compounds.
The `p`/`<` pair alone is a very strong RESULTS indicator (notebook 02, C7), so losing it would be harmful.

### 1.1 Stopwords — and why clinical negations must survive

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
removed_by_sklearn = sorted(w for w in ENGLISH_STOP_WORDS if w in NEGATIONS or w in {"more", "less", "before", "after", "between", "per", "over", "under"})
print("sklearn stopwords that MediScope KEEPS:", removed_by_sklearn)
neg = "There was no significant difference and the drug did not reduce pain without side effects"
print("\nsklearn removal :", " ".join(w for w in neg.lower().split() if w not in ENGLISH_STOP_WORDS))
print("MediScope nostop:", preprocess(neg, "nostop"))

Standard stopword removal turns *"no significant difference"* into *"significant difference"* — the **opposite** claim. MediScope's list is the scikit-learn list **minus** negations
and a few clinically meaningful function words (*more, less, before, after, between, per, over, under, high, low* …).

### 1.2 Stemming vs lemmatisation

In [ ]:
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
try:
    nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
    lem = WordNetLemmatizer(); lem.lemmatize("studies"); have_wn = True
except Exception as e:
    have_wn = False; print("WordNet unavailable:", e)
ps = PorterStemmer()
words = ["patients", "studies", "randomized", "randomised", "treatments", "improved", "improvement", "infections", "diagnosis", "diagnoses", "carcinoma", "carcinomas", "trials", "was"]
tab = pd.DataFrame({"word": words, "Porter stem": [ps.stem(w) for w in words]})
if have_wn: tab["WordNet lemma (noun)"] = [lem.lemmatize(w) for w in words]; tab["WordNet lemma (verb)"] = [lem.lemmatize(w, "v") for w in words]
tab.set_index("word").T

Stemming conflates `randomized/randomised` (good) but also produces non-words (`improv`, `diagnos`) and merges clinically distinct forms (*diagnosis*/*diagnoses*), whereas
lemmatisation is more conservative but needs a POS tag and an external corpus download. For the deployed pipeline we avoid a corpus dependency: **stemming is available as an
experiment strategy; lemmatisation is shown here but not shipped** (its marginal benefit for sentence-role classification is tested below through the stem strategy as a proxy).

### 1.3 Medical abbreviations, dosages and regex preprocessing

In [ ]:
from src.preprocessing import ABBREVIATIONS
print("curated abbreviation map (illustrative):", dict(list(ABBREVIATIONS.items())[:6]), "...")
for s in examples:
    print("\n", s); print("  dosages  :", extract_dosages(s)); print("  clean    :", preprocess(s, "clean")); print("  numshape :", preprocess(s, "numshape")); print("  expand   :", preprocess(s, "expand"))

The `expand` strategy shows the **ambiguity risk** of abbreviation expansion: `OR` (odds ratio vs the conjunction *or*) and `HR` (hazard ratio vs heart rate / hour) cannot be resolved without context.
A hand-curated map is therefore limited to unambiguous cases and is evaluated, not assumed beneficial.

## 2. Comparing preprocessing strategies

| Strategy | What changes |
|---|---|
| `raw` | lowercase only (tokens produced by scikit-learn's default pattern would be lossy, so we keep whitespace tokens) |
| `clean` | Unicode/whitespace normalisation + regex tokenisation keeping numbers, units, operators, negations |
| `numshape` | `clean` + every number → `num` / `pct` |
| `nostop` | `clean` + stopword removal (negations kept) |
| `stem` | `clean` + Porter stemming |
| `expand` | `clean` + abbreviation expansion |

Protocol: TF-IDF (1,2)-grams, `min_df=2`, sublinear TF → Logistic Regression (`C=5`). Each strategy is trained on **3 different stratified 30k-sentence subsets** of the
training split and scored on the full **dev** split. We report mean ± std of macro-F1 so differences can be judged against seed noise.

In [ ]:
SEEDS = [0, 1, 2]; N_TRAIN = 30_000
def run(strategy, seed, ngram=(1, 2), min_df=2, sublinear=True, max_features=200_000):
    sub = D.stratified_subset(train, N_TRAIN, seed)
    vec = build_tfidf(strategy, ngram, max_features, min_df, sublinear)
    t0 = time.time(); Xtr = vec.fit_transform(sub.text); fit_vec = time.time() - t0
    clf = LogisticRegression(C=5, max_iter=300, solver="lbfgs", n_jobs=1).fit(Xtr, sub.label)
    pred = clf.predict(vec.transform(dev.text))
    m = classification_metrics(dev.label, pred, LAB)
    return {"strategy": strategy, "seed": seed, "macro_f1": m["f1_macro"], "accuracy": m["accuracy"], "n_features": Xtr.shape[1], "vec_seconds": fit_vec}
res = pd.DataFrame([run(s, sd) for s in STRATEGIES for sd in SEEDS])
summary = res.groupby("strategy").agg(macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"), acc_mean=("accuracy", "mean"), n_features=("n_features", "mean"), vec_s=("vec_seconds", "mean")).sort_values("macro_f1_mean", ascending=False)
summary.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
s = summary.sort_values("macro_f1_mean")
ax.barh(s.index, s.macro_f1_mean, xerr=s.macro_f1_std, color=PALETTE[0], ecolor=PALETTE[1], capsize=3)
ax.set_xlim(s.macro_f1_mean.min() - .02, s.macro_f1_mean.max() + .01); ax.set_xlabel("dev macro-F1 (mean ± std, 3 subsets)"); ax.set_title("Preprocessing strategy comparison")
plt.tight_layout(); plt.show()
best = summary.index[0]; base = summary.loc["clean"]; gap = summary.macro_f1_mean.iloc[0] - summary.macro_f1_mean.iloc[1]
note(f"Best strategy by mean macro-F1: **{best}** ({summary.macro_f1_mean.iloc[0]:.4f}); `clean` scored {base.macro_f1_mean:.4f}. The margin between the best two is {gap:.4f} "
     f"versus seed-to-seed std ≈ {summary.macro_f1_std.mean():.4f}, so differences below ≈{2*summary.macro_f1_std.mean():.3f} should be treated as ties. "
     f"Worst: **{summary.index[-1]}** ({summary.macro_f1_mean.iloc[-1]:.4f}).")

### 2.1 Statistical check

Paired by seed, is any strategy reliably different from `clean`? With only 3 pairs a *t*-test has little power, so this is reported as a sanity check, not as proof.

In [ ]:
from scipy.stats import ttest_rel
piv = res.pivot(index="seed", columns="strategy", values="macro_f1")
rows = [{"strategy": s, "mean_diff_vs_clean": (piv[s] - piv["clean"]).mean(), "paired_t_p": (ttest_rel(piv[s], piv["clean"]).pvalue if s != "clean" else np.nan)} for s in STRATEGIES]
pd.DataFrame(rows).set_index("strategy").round(4)

## 3. Feature-engineering ablations (strategy fixed to `clean`)

We vary one TF-IDF setting at a time: n-gram range, sublinear TF, `min_df`, and plain term counts.

In [ ]:
configs = {
    "unigrams":                dict(ngram=(1, 1)),
    "uni+bigrams (default)":   dict(ngram=(1, 2)),
    "uni+bi+trigrams":         dict(ngram=(1, 3)),
    "no sublinear TF":         dict(sublinear=False),
    "min_df=1":                dict(min_df=1),
    "min_df=5":                dict(min_df=5),
    "max_features=20k":        dict(max_features=20_000),
}
rows = []
for name, kw in configs.items():
    rs = [run("clean", sd, **kw) for sd in SEEDS]
    rows.append({"config": name, "macro_f1": np.mean([r["macro_f1"] for r in rs]), "std": np.std([r["macro_f1"] for r in rs], ddof=1), "n_features": int(np.mean([r["n_features"] for r in rs]))})
abl = pd.DataFrame(rows).set_index("config"); abl.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
a = abl.sort_values("macro_f1"); axes[0].barh(a.index, a.macro_f1, xerr=a["std"], color=PALETTE[3], ecolor=PALETTE[1], capsize=3); axes[0].set_xlim(a.macro_f1.min() - .01, a.macro_f1.max() + .01); axes[0].set_title("dev macro-F1")
axes[1].barh(a.index, a.n_features, color=PALETTE[2]); axes[1].set_xscale("log"); axes[1].set_title("number of features (log)"); axes[1].set_yticklabels([])
plt.tight_layout(); plt.show()
u, b, t = abl.loc["unigrams", "macro_f1"], abl.loc["uni+bigrams (default)", "macro_f1"], abl.loc["uni+bi+trigrams", "macro_f1"]
note(f"Adding bigrams changes macro-F1 by {b - u:+.4f} over unigrams; adding trigrams changes it by a further {t - b:+.4f} while multiplying the feature count "
     f"({int(abl.loc['uni+bigrams (default)', 'n_features']):,} → {int(abl.loc['uni+bi+trigrams', 'n_features']):,}). "
     f"Sublinear TF changes the score by {abl.loc['uni+bigrams (default)', 'macro_f1'] - abl.loc['no sublinear TF', 'macro_f1']:+.4f}. The cost–benefit of each option is explicit in the table above.")

## 4. TF-IDF vs Word2Vec document embeddings

We train a skip-gram **Word2Vec** (100-d, `window=5`, `min_count=3`) on the full *training* text (self-supervised — no labels), then represent each sentence as the **mean of its word vectors**
and train the same Logistic Regression. This isolates *representation* quality (sparse lexical vs dense distributional).

In [ ]:
from src.features import train_word2vec, doc_vectors
t0 = time.time(); w2v = train_word2vec(train.text, vector_size=100, window=5, min_count=3, epochs=8, workers=1); w2v_time = time.time() - t0
print(f"Word2Vec trained in {w2v_time:.0f}s | vocabulary {len(w2v.wv):,}")
for w in ["placebo", "metformin", "hypertension", "randomized"]:
    print(w, "->", [x for x, _ in w2v.wv.most_similar(w, topn=6)])
w2v.save(str(config.DATA_PROCESSED / "word2vec.model"))

Nearest neighbours should be *domain-sensible* (drugs near drugs, conditions near conditions). This is a qualitative validity check of the embedding before using it.

In [ ]:
sub = D.stratified_subset(train, N_TRAIN, 0)
Xtr_w, Xdev_w = doc_vectors(w2v, sub.text), doc_vectors(w2v, dev.text)
clf_w = LogisticRegression(C=5, max_iter=500).fit(Xtr_w, sub.label)
m_w = classification_metrics(dev.label, clf_w.predict(Xdev_w), LAB)
tfidf_f1 = abl.loc["uni+bigrams (default)", "macro_f1"]
cmp_ = pd.DataFrame({"representation": ["TF-IDF (1,2)-gram, sparse", "Word2Vec mean vector, 100-d dense"], "dev macro-F1": [tfidf_f1, m_w["f1_macro"]], "dimensions": [int(abl.loc["uni+bigrams (default)", "n_features"]), 100]}).set_index("representation")
display(cmp_.round(4))
note(f"TF-IDF reaches macro-F1 {tfidf_f1:.4f} versus {m_w['f1_macro']:.4f} for averaged Word2Vec vectors (Δ = {tfidf_f1 - m_w['f1_macro']:+.4f}). "
     f"Averaging discards word order and negation scope, and the 100-d bottleneck loses rare discriminative tokens (`p`, `<`, `mg`); sparse n-grams keep them. "
     f"Embeddings are therefore **not** used for the classical models, but they are re-used to initialise the BiLSTM embedding layer in notebook 05, where order is modelled by the network.")

## 5. Vocabulary and OOV analysis per strategy

In [ ]:
rows = []
for strat in STRATEGIES:
    tr_tok = [preprocess(t, strat).split() for t in train.text.sample(60000, random_state=0)]
    dv_tok = [preprocess(t, strat).split() for t in dev.text.sample(10000, random_state=0)]
    cnt = Counter(w for s in tr_tok for w in s); vocab = {w for w, c in cnt.items() if c >= 2}
    toks = [w for s in dv_tok for w in s]
    rows.append({"strategy": strat, "train_vocab (freq>=2)": len(vocab), "types_total": len(cnt), "dev_OOV_token_rate": np.mean([w not in vocab for w in toks]), "avg_tokens_per_sentence": np.mean([len(s) for s in dv_tok])})
voc = pd.DataFrame(rows).set_index("strategy"); voc.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
voc["train_vocab (freq>=2)"].plot.bar(ax=axes[0], color=PALETTE[0]); axes[0].set_title("Vocabulary size (freq ≥ 2)"); axes[0].tick_params(axis="x", rotation=30)
(voc.dev_OOV_token_rate * 100).plot.bar(ax=axes[1], color=PALETTE[1]); axes[1].set_title("Dev OOV token rate (%)"); axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
note(f"`numshape` shrinks the vocabulary the most ({voc.loc['numshape', 'train_vocab (freq>=2)']:,} vs {voc.loc['clean', 'train_vocab (freq>=2)']:,} for `clean`) and lowers dev OOV to "
     f"{voc.loc['numshape', 'dev_OOV_token_rate']:.2%} (vs {voc.loc['clean', 'dev_OOV_token_rate']:.2%}), because most unseen tokens are numbers. `stem` also reduces vocabulary ({voc.loc['stem', 'train_vocab (freq>=2)']:,}). "
     f"Whether the smaller vocabulary translates into better F1 is answered by section 2.")

## 6. Decision

The strategy and feature configuration for the **classical pipeline** are chosen from the dev-set experiments above (best mean macro-F1; ties resolved in favour of the simpler option).
The choice is written to disk so notebook 04 reads it instead of hard-coding it, keeping the project reproducible.

In [ ]:
from src.utils import save_json
tie = summary[summary.macro_f1_mean >= summary.macro_f1_mean.max() - 2 * summary.macro_f1_std.mean()]
simplicity = ["clean", "raw", "numshape", "nostop", "stem", "expand"]
chosen = next(s for s in simplicity if s in tie.index)
best_ng = abl.loc[["unigrams", "uni+bigrams (default)", "uni+bi+trigrams"], "macro_f1"]
ngram = {"unigrams": (1, 1), "uni+bigrams (default)": (1, 2), "uni+bi+trigrams": (1, 3)}[best_ng.idxmax() if best_ng.max() - b > .002 else "uni+bigrams (default)"]
choice = {"strategy": chosen, "ngram_range": list(ngram), "min_df": 2, "sublinear_tf": True, "max_features": 200000,
          "evidence": {"strategy_macro_f1": summary.macro_f1_mean.round(4).to_dict(), "tie_band": round(2 * float(summary.macro_f1_std.mean()), 4),
                       "ngram_ablation": abl.macro_f1.round(4).to_dict(), "tfidf_vs_word2vec": {"tfidf": round(float(tfidf_f1), 4), "word2vec_mean": round(float(m_w["f1_macro"]), 4)}}}
save_json(choice, config.ART_CLS / "preprocessing_choice.json")
display(Markdown(f"**Chosen:** strategy = `{chosen}`, n-grams = {ngram}, `min_df=2`, sublinear TF. Strategies within the tie band: {list(tie.index)}."))

## Conclusions

* Generic NLP defaults (default tokeniser, stopword removal, aggressive stemming) can destroy clinically meaningful signal: dropped negations, split decimals/dosages, lost `p <` patterns.
* Strategy differences are measured against seed noise; ties are resolved toward the simpler pipeline.
* Sparse n-gram TF-IDF is compared directly to averaged Word2Vec vectors; the measured gap determines the role of embeddings in the project (BiLSTM initialisation).
* The decision is persisted (`preprocessing_choice.json`) and consumed by notebook 04.